# Chapter 7 Functions as First-Class Objects

## Topic 1: Treating a Function Like an Object (First-Class Functions)

### Concept
In Python, functions are first-class objects. A "first-class object" is a program entity that can be:
1. Created at runtime (via `def` or `lambda`).
2. Assigned to a variable or element in a data structure.
3. Passed as an argument to a function.
4. Returned as the result of a function.

### Underlying Mechanism
- Functions in Python are instances of the `function` class (`type(fn)` returns `<class 'function'>`).
- Assigning a function to a new variable creates an alias pointing to the exact same function object in memory.
- Function objects possess standard attributes, such as:
  - `__doc__`: Holds the docstring text used by `help()`.
  - `__name__`: Holds the string name defined in the `def` statement.
- Custom attributes can be dynamically attached directly to a function object (e.g., `fn.my_attr = 10`).

### Gotchas
* **Alias State Sharing:** Assigning a function to another variable name creates an alias pointing to the exact same object in memory [passage 7]. Dynamically attaching or mutating custom attributes on the alias mutates the underlying function object (e.g., `fact = factorial; fact.custom_attr = 42` means `factorial.custom_attr` is also `42`).
* **Missing Docstrings:** The `__doc__` attribute retrieves the string literal declared at the top of the function body. If no docstring is declared, `__doc__` evaluates to `None`.

In [2]:
def square(x):
    """Calculates square of x."""
    return x * x

sq_alias = square
sq_alias.custom_attr = 42

print(1, sq_alias(4), sq_alias.__name__, sq_alias.__doc__.strip()) # 16, square, "Calculates square of x"
print(2, square.custom_attr) # 42
print(sq_alias.__doc__)

1 16 square Calculates square of x.
2 42
Calculates square of x.


## Topic 2: Higher-Order Functions & Modern Replacements

### Concept
A higher-order function is a function that either takes a function as an argument or returns a function as a result (e.g., `sorted(..., key=len)`, `map`, `filter`, `reduce`).

### Modern Replacements in Python 3
1. `map` and `filter`:
   - In Python 3, `map` and `filter` return **generator iterators**.
   - List comprehensions (`[expr for x in seq if cond]`) and generator expressions replace the combination of `map` and `filter` with superior readability and without requiring `lambda`.
2. `reduce`:
   - Demoted from a built-in (Python 2) to `functools.reduce` in Python 3.
   - For summation, the built-in `sum(iterable)` is cleaner and faster.
3. Terminal Reducing Built-ins:
   - `all(iterable)`: Returns `True` if no element is falsy. Note: `all([])` evaluates to `True`.
   - `any(iterable)`: Returns `True` if any element is truthy. Note: `any([])` evaluates to `False`.

In [1]:
from functools import reduce
from operator import add

data = [1, 2, 3, 4]

res_reduce = reduce(add, data, 10) # 10 is the initial initializer
res_all = all([x > 0 for x in data])
res_any_empty = any([])
res_all_empty = all([])

print(1, res_reduce, res_all) # 20, True
print(2, res_any_empty, res_all_empty) # False, True

1 20 True
2 False True


## Topic 3: Anonymous Functions (`lambda`)

### Concept & Syntax
The `lambda` keyword constructs an anonymous function object inline within an expression:
`lambda param1, param2: expression`

### Restrictions & Mechanism
- Syntactic Restriction: The body of a `lambda` MUST be a single pure Python expression.
- Statements such as `while`, `try`, `raise`, `pass`, or assignment (`=`) cannot occur inside a `lambda`.
- Default parameter values and variable argument forms (`*args`, `**kwargs`) are fully supported in `lambda` definitions.
- Primary Use Case: Small, one-off key functions passed directly to higher-order functions (e.g., `sorted(words, key=lambda w: w[::-1])`).

In [2]:
f = lambda x, y=10: x + y

print(1, f(5)) # 15
print(2, f(5, 20)) # 25

1 15
2 25


## Topic 4: The Nine Flavors of Callable Objects

### Concept
To determine if an object can be invoked using the call operator `()`, use the built-in `callable(obj)` function.

### The Nine Callable Types in Python Data Model
1. User-defined functions: Created with `def` or `lambda`.
2. Built-in functions: C-implemented functions in CPython (e.g., `len`, `abs`).
3. Built-in methods: C-implemented methods (e.g., `dict.get`).
4. Methods: Functions defined inside a class body.
5. Classes: Invoking a class runs `__new__` to create the instance, then `__init__` to initialize it.
6. Class instances: Instances of classes that implement a `__call__` method.
7. Generator functions: Functions/methods using `yield`; calling them returns a generator object.
8. Native coroutine functions: Defined with `async def`; calling them returns a coroutine object.
9. Asynchronous generator functions: Defined with `async def` containing `yield`.

In [3]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

m = Multiplier(2)

print(1, callable(m), callable(Multiplier), callable(abs)) # False, True, True

1 False True True


## Topic 5: User-Defined Callable Types (`__call__`)

### Concept & Mechanism
- Any arbitrary Python class can produce function-like callable objects by defining an instance method named `__call__`.
- Invoking `instance(*args, **kwargs)` is syntactic sugar for calling `instance.__call__(*args, **kwargs)`.
- Use Case: Creating stateful function-like objects that maintain internal state across invocations (e.g., memoization/caching, randomized samplers) without relying on global variables.

In [4]:

class StatefulMultiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, x):
        self.factor += 1
        return x * self.factor

m = StatefulMultiplier(2)

print(1, m(10)) # 30
print(2, m(10)) # 40


1 30
2 40


## Topic 6: Positional-Only and Keyword-Only Parameters

### Parameter Modes & Boundaries
1. By default, when there is no positional and keyword enforcement, an argument can be passed in either positionally or as a keyword argument.
1. Positional-Only Parameters (`/`):
   - Parameters declared **BEFORE** `/` in a signature MUST be supplied positionally.
   - Passing them as keyword arguments raises a `TypeError`.
   - This mainly solves three problems:
      1. Freedom to rename parameter names without breaking API contracts; a keyword argument becomes part of the public API and renaming such an argument breaks the client code.
      2. Avoiding keword clashes with `**kwargs` which allows an arbitrary keyword arguments, a named parameter risks colliding with a dictionary key that the caller passes.
      3. Semantic Meaninglessness and Build-in Consistency: For many mathematical or fundamental utility functions, parameter names add no semantic value (e.g., `abs(x)` or `len(obj)`). In Python's C-implemented built-ins, these have always been positional-only (`abs(x=5)` raises TypeError). The / syntax brings this same capability to pure Python functions.
2. Keyword-Only Parameters (`*` or after `*args`):
   - Parameters declared AFTER `*` or AFTER `*args` MUST be supplied as keyword arguments.
   - A bare `*` in a signature specifies that no variable positional arguments are accepted, but subsequent parameters are keyword-only.
3. Unpacking:
   - `*iterable` unpacks sequence items into positional arguments.
   - `**mapping` unpacks dictionary key-value pairs into keyword arguments.



In [8]:
def f(a, b):
    print(a, b)

f(1, 2)
f(1, b=2)
f(b=2, a=1)

1 2
1 2
1 2


In [7]:
def combo(a, b, /, c, *, d=100):
    return a + b + c + d

print(1, combo(1, 2, 3, d=4)) # 10
print(2, combo(1, 2, c=3)) # 106
#What happens if you execute combo(a=1, b=2, c=3)? # TypeError
#combo(a=1, b=2, c=3)

1 10
2 106


In [14]:
def func(*args, **kwargs):
    print(type(args), args)
    if kwargs:
        print(kwargs)

func(1,2,3)
func(10, 20, 30, a=1, b=2)

<class 'tuple'> (1, 2, 3)
<class 'tuple'> (10, 20, 30)
{'a': 1, 'b': 2}


## Topic 7: Packages for Functional Programming (`operator` & `functools`)

### The `operator` Module
Replaces simple lambda functions with optimized C-implemented functions:
- `operator.add`, `operator.mul`, `operator.sub`: Function equivalents of operators.
- `itemgetter(a, b, ...)`: Factory that returns a callable extracting items via `[]`. Passing multiple indices returns a **tuple** of extracted items.
- `attrgetter('a.b')`: Factory that returns a callable extracting object attributes. Supports dotted paths to navigate nested attributes.
- `methodcaller(name, *args)`: Factory returning a callable that invokes a method by string name on its target object, pre-binding any additional arguments provided.

### `functools.partial`
- `functools.partial(func, *args, **keywords)`: Constructs a new callable with specific positional and keyword arguments frozen.
- Attributes on partial objects:
  - `.func`: Accesses the underlying target function.
  - `.args`: Returns a tuple of frozen positional arguments.
  - `.keywords`: Returns a dictionary of frozen keyword arguments.


In [ ]:

from operator import itemgetter, methodcaller
from functools import partial

# Itemgetter
get_first_and_last = itemgetter(0, -1)
names = ["Alice", "Bob", "Charlie"]

# Methodcaller
replace_spaces = methodcaller("replace", " ", "_")

# Partial
def power(base, exponent):
    return base ** exponent

square_fn = partial(power, exponent=2)
cube_fn = partial(power, 3) # Freeze first positional argument (base=3)

print(1, get_first_and_last(names)) # (Alice, Charlie)
print(2, replace_spaces("hello world python")) #hello_world_python
print(3, square_fn(5), cube_fn(3), square_fn.keywords) # 25, 27, {'exponent': 2}


1 ('Alice', 'Charlie')
2 hello_world_python
3 25 27 {'exponent': 2}
